In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [ ]:
INPUT_STAGE1 = "stage1_outputs/prepared_stage1_dataset.csv"
INPUT_PRED = "stage3_model_outputs/buyout_predictions.csv"

OUTPUT_DIR = Path("stage4_financial_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_ORDERS_REGION = 30
MIN_ORDERS_DELIVERY = 30
MIN_ORDERS_MANAGER = 50
MIN_ORDERS_REASON = 20
SPEED_THRESHOLD_DAYS = 3
RISK_THRESHOLD = 0.7

In [ ]:
def load_bool_column(series):
    vals = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .replace({"<na>": None, "nan": None, "": None})
    )
    return vals.map({"true": True, "false": False, "1": True, "0": False}).astype("boolean")


def load_data(stage1_path, pred_path):
    df = pd.read_csv(stage1_path)

    for col in ["buyout_flag", "outcome_unknown", "lifecycle_incomplete"]:
        if col in df.columns:
            df[col] = load_bool_column(df[col])

    numeric_cols = [
        "speed_to_delivery_days",
        "lead_price",
        "order_amount",
        "delivery_cost"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    pred_df = None
    pred_path = Path(pred_path)
    if pred_path.exists():
        pred_df = pd.read_csv(pred_path)

    return df, pred_df


def prepare_financial_dataset(df, pred_df=None):
    stats = {"rows_input": int(len(df))}

    if "lifecycle_incomplete" in df.columns:
        bad = df["lifecycle_incomplete"].astype("boolean").fillna(False)
        stats["excluded_lifecycle_incomplete"] = int(bad.sum())
        df = df.loc[~bad].copy()
    else:
        stats["excluded_lifecycle_incomplete"] = 0

    if "outcome_unknown" in df.columns:
        unknown = df["outcome_unknown"].astype("boolean").fillna(False)
        stats["excluded_outcome_unknown"] = int(unknown.sum())
        df = df.loc[~unknown].copy()
    else:
        stats["excluded_outcome_unknown"] = 0

    df = df.loc[df["buyout_flag"].notna()].copy()
    df["is_buyout"] = df["buyout_flag"].astype(int)
    df["is_non_buyout"] = 1 - df["is_buyout"]

    order_value = pd.Series(np.nan, index=df.index)

    if "lead_price" in df.columns:
        order_value = pd.to_numeric(df["lead_price"], errors="coerce")

    if "order_amount" in df.columns:
        amount = pd.to_numeric(df["order_amount"], errors="coerce")
        order_value = order_value.where(order_value.notna(), amount)

    median_order_value = float(order_value.dropna().median()) if order_value.notna().any() else 10000.0
    df["order_value_est"] = order_value.fillna(median_order_value)

    df["estimated_loss"] = np.where(df["is_non_buyout"] == 1, df["order_value_est"], 0.0)

    if pred_df is not None and "lead_id" in pred_df.columns:
        keep_cols = [c for c in ["lead_id", "buyout_probability", "predicted_class", "model_name"] if c in pred_df.columns]
        df = df.merge(pred_df[keep_cols], on="lead_id", how="left")

    stats["rows_final"] = int(len(df))
    stats["buyout_rate"] = float(df["is_buyout"].mean()) if len(df) else None
    stats["median_order_value"] = median_order_value

    return df, stats


df_raw, pred_df = load_data(INPUT_STAGE1, INPUT_PRED)
df, prep_stats = prepare_financial_dataset(df_raw, pred_df)

print(json.dumps(prep_stats, ensure_ascii=False, indent=2))
print(df.shape)
df.head(3)

In [ ]:
overall_summary = {
    "orders_total": int(len(df)),
    "buyout_orders": int(df["is_buyout"].sum()),
    "non_buyout_orders": int(df["is_non_buyout"].sum()),
    "buyout_rate": float(df["is_buyout"].mean()),
    "total_estimated_loss": float(df["estimated_loss"].sum()),
    "avg_loss_per_non_buyout": float(df.loc[df["is_non_buyout"] == 1, "estimated_loss"].mean()),
    "median_order_value": float(df["order_value_est"].median())
}

overall_summary

In [ ]:
def loss_by_segment(df, group_col, min_orders=30):
    if group_col not in df.columns:
        return pd.DataFrame()

    temp = df.copy()
    temp[group_col] = temp[group_col].fillna("Не указан")

    out = (
        temp.groupby(group_col)
        .agg(
            orders=("lead_id", "count"),
            buyouts=("is_buyout", "sum"),
            non_buyouts=("is_non_buyout", "sum"),
            buyout_rate=("is_buyout", "mean"),
            total_loss=("estimated_loss", "sum"),
            avg_loss_per_order=("estimated_loss", "mean"),
            avg_loss_per_non_buyout=("estimated_loss", lambda x: x[x > 0].mean() if (x > 0).any() else 0.0)
        )
        .reset_index()
    )

    out = out[out["orders"] >= min_orders].copy()
    out = out.sort_values(["total_loss", "orders"], ascending=[False, False])
    return out


loss_region = loss_by_segment(df, "region", min_orders=MIN_ORDERS_REGION)
loss_delivery = loss_by_segment(df, "delivery_service", min_orders=MIN_ORDERS_DELIVERY)
loss_manager = loss_by_segment(df, "manager_id", min_orders=MIN_ORDERS_MANAGER)
loss_reason = loss_by_segment(df, "rejection_category", min_orders=MIN_ORDERS_REASON)

loss_region.head(10)

In [ ]:
loss_region.to_csv(OUTPUT_DIR / "loss_by_region.csv", index=False)
loss_delivery.to_csv(OUTPUT_DIR / "loss_by_delivery.csv", index=False)
loss_manager.to_csv(OUTPUT_DIR / "loss_by_manager.csv", index=False)
loss_reason.to_csv(OUTPUT_DIR / "loss_by_rejection_reason.csv", index=False)

In [ ]:
speed_df = df[df["speed_to_delivery_days"].notna()].copy()

fast = speed_df[speed_df["speed_to_delivery_days"] <= SPEED_THRESHOLD_DAYS].copy()
slow = speed_df[speed_df["speed_to_delivery_days"] > SPEED_THRESHOLD_DAYS].copy()

fast_buyout_rate = float(fast["is_buyout"].mean()) if len(fast) else np.nan
slow_buyout_rate = float(slow["is_buyout"].mean()) if len(slow) else np.nan
delta_buyout = fast_buyout_rate - slow_buyout_rate if pd.notna(fast_buyout_rate) and pd.notna(slow_buyout_rate) else np.nan

potential_saved_orders = delta_buyout * len(slow) if pd.notna(delta_buyout) else np.nan
potential_saved_revenue = potential_saved_orders * prep_stats["median_order_value"] if pd.notna(potential_saved_orders) else np.nan

speed_impact = {
    "threshold_days": SPEED_THRESHOLD_DAYS,
    "fast_orders": int(len(fast)),
    "slow_orders": int(len(slow)),
    "fast_buyout_rate": fast_buyout_rate,
    "slow_buyout_rate": slow_buyout_rate,
    "delta_buyout_rate": delta_buyout,
    "potential_saved_orders": float(potential_saved_orders) if pd.notna(potential_saved_orders) else None,
    "potential_saved_revenue": float(potential_saved_revenue) if pd.notna(potential_saved_revenue) else None
}

speed_impact

In [ ]:
risk_summary = {}

if "buyout_probability" in df.columns:
    high_risk = df[df["buyout_probability"] < RISK_THRESHOLD].copy()
    low_risk = df[df["buyout_probability"] >= RISK_THRESHOLD].copy()

    risk_summary = {
        "risk_threshold": RISK_THRESHOLD,
        "high_risk_orders": int(len(high_risk)),
        "low_risk_orders": int(len(low_risk)),
        "high_risk_buyout_rate": float(high_risk["is_buyout"].mean()) if len(high_risk) else None,
        "low_risk_buyout_rate": float(low_risk["is_buyout"].mean()) if len(low_risk) else None,
        "high_risk_total_loss": float(high_risk["estimated_loss"].sum()) if len(high_risk) else None,
        "low_risk_total_loss": float(low_risk["estimated_loss"].sum()) if len(low_risk) else None,
    }

risk_summary

In [ ]:
def plot_loss_bar(table, group_col, title, output_path, top_n=10):
    if table.empty:
        return

    temp = table.head(top_n).copy().iloc[::-1]

    plt.figure(figsize=(12, 7))
    plt.barh(temp[group_col].astype(str), temp["total_loss"])
    plt.xlabel("Оценка потерь")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()


plot_loss_bar(
    loss_region, "region",
    "Топ регионов по оценке потерь",
    OUTPUT_DIR / "loss_region.png",
    top_n=10
)

plot_loss_bar(
    loss_delivery, "delivery_service",
    "Топ служб доставки по оценке потерь",
    OUTPUT_DIR / "loss_delivery.png",
    top_n=10
)

plot_loss_bar(
    loss_manager, "manager_id",
    "Топ менеджеров по оценке потерь",
    OUTPUT_DIR / "loss_manager.png",
    top_n=10
)

plot_loss_bar(
    loss_reason, "rejection_category",
    "Топ причин отказа по оценке потерь",
    OUTPUT_DIR / "loss_reason.png",
    top_n=10
)

In [ ]:
scenario_df = pd.DataFrame([{
    "scenario": f"Снизить долю заказов со speed_to_delivery_days > {SPEED_THRESHOLD_DAYS}",
    "current_slow_orders": speed_impact["slow_orders"],
    "current_slow_buyout_rate": speed_impact["slow_buyout_rate"],
    "target_buyout_rate": speed_impact["fast_buyout_rate"],
    "potential_saved_orders": speed_impact["potential_saved_orders"],
    "potential_saved_revenue": speed_impact["potential_saved_revenue"]
}])

scenario_df.to_csv(OUTPUT_DIR / "financial_scenario_effect.csv", index=False)
scenario_df

In [ ]:
with open(OUTPUT_DIR / "financial_overall_summary.json", "w", encoding="utf-8") as f:
    json.dump(overall_summary, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "financial_speed_impact.json", "w", encoding="utf-8") as f:
    json.dump(speed_impact, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "financial_risk_summary.json", "w", encoding="utf-8") as f:
    json.dump(risk_summary, f, ensure_ascii=False, indent=2)

In [ ]:
business_summary = {
    "orders_total": overall_summary["orders_total"],
    "non_buyout_orders": overall_summary["non_buyout_orders"],
    "buyout_rate": overall_summary["buyout_rate"],
    "total_estimated_loss": overall_summary["total_estimated_loss"],
    "avg_loss_per_non_buyout": overall_summary["avg_loss_per_non_buyout"],
    "largest_region_loss": float(loss_region.iloc[0]["total_loss"]) if not loss_region.empty else None,
    "largest_delivery_loss": float(loss_delivery.iloc[0]["total_loss"]) if not loss_delivery.empty else None,
    "largest_manager_loss": float(loss_manager.iloc[0]["total_loss"]) if not loss_manager.empty else None,
    "speed_threshold_days": SPEED_THRESHOLD_DAYS,
    "potential_saved_orders": speed_impact["potential_saved_orders"],
    "potential_saved_revenue": speed_impact["potential_saved_revenue"],
}

with open(OUTPUT_DIR / "financial_business_summary.json", "w", encoding="utf-8") as f:
    json.dump(business_summary, f, ensure_ascii=False, indent=2)

print(json.dumps(business_summary, ensure_ascii=False, indent=2))